# 🎬 YouTube Frame Captioning (LLaVA-NeXT-Video)

이 노트북은 YouTube 영상의 하이라이트 프레임들을 **LLaVA-NeXT-Video** 모델을 사용해 캡셔닝합니다.

### 📁 폴더 구조 가이드 (Google Drive)
사용자 지정 경로에 맞춰 드라이브에 데이터를 미리 업로드해 주세요.

```
MyDrive/
└── tzudong/
    └── frame-caption/
        ├── scripts/                # 이 노트북 파일 위치
        ├── data/
        │   └── frames/             # [입력] 프레임 이미지 폴더
        │       └── {video_id}/ ...
        └── frame-caption/          # [출력] 결과가 저장될 폴더 (자동 생성됨)
            └── {video_id}.jsonl
```

## 1. 환경 설정 & 라이브러리 설치

In [ ]:
# 필요 라이브러리 설치
!pip install -q transformers accelerate
!pip install -q pillow

## 2. Google Drive 마운트 및 경로 설정

In [ ]:
from google.colab import drive
import os

# 구글 드라이브 마운트
drive.mount("/content/drive")

# [사용자 지정 경로 설정]
# 기본 경로: MyDrive/tzudong/frame-caption
BASE_DIR = "/content/drive/MyDrive/tzudong/frame-caption"

# 입력 경로 (data/frames)
FRAMES_DIR = os.path.join(BASE_DIR, "data", "frames")

# 출력 경로 (frame-caption) - 로컬 구조와 동일하게 맞춤
OUTPUT_DIR = os.path.join(BASE_DIR, "frame-caption")

# 출력 디렉토리 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📂 작업 기본 경로: {BASE_DIR}")
print(f"📂 입력 경로(Frames): {FRAMES_DIR}")
print(f"📂 출력 경로(Output): {OUTPUT_DIR}")

# 경로 확인
if not os.path.exists(FRAMES_DIR):
    print("⚠️ 경고: 입력 프레임 폴더를 찾을 수 없습니다. 경로를 확인해주세요.")

## 3. 모델 로드 (LLaVA-NeXT-Video)
Colab Pro/Pro+ (A100/L4 등) 환경이므로 **Float16** 원본 정밀도를 사용하여 최고의 품질을 확보합니다.
(양자화 없음)

In [ ]:
import torch
from transformers import LlavaNextVideoProcessor, LlavaNextVideoForConditionalGeneration

MODEL_ID = "llava-hf/LLaVA-NeXT-Video-7B-hf"

print("🚀 모델 로딩 중... (최적 성능 모드)")

# 프로세서 로드
processor = LlavaNextVideoProcessor.from_pretrained(MODEL_ID)

# 모델 로드 (Float16)
model = LlavaNextVideoForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True
)

print("✅ 모델 로드 완료!")

## 4. 유틸리티 함수 정의
- **`extract_json`**: LLM 출력에서 JSON 부분만 안정적으로 추출합니다.
- **`generate_caption`**: 모델 추론을 수행합니다.

In [ ]:
import json
import re
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm


def parse_segment_folder(folder_name):
    """폴더명에서 메타데이터 파싱 (순번_시작초_끝초)"""
    match = re.match(r"^(\d+)_(\d+)_(\d+)$", folder_name)
    if match:
        return {
            "rank": int(match.group(1)),
            "start_sec": int(match.group(2)),
            "end_sec": int(match.group(3)),
        }
    return None


def load_frames(folder_path):
    """폴더 내의 webp/jpg 이미지 로드"""
    folder = Path(folder_path)
    # webp 우선 검색, 없으면 jpg
    files = list(folder.glob("*.webp")) + list(folder.glob("*.jpg"))
    files.sort(key=lambda x: int(x.stem))

    images = []
    for f in files:
        try:
            img = Image.open(f).convert("RGB")
            images.append(img)
        except Exception as e:
            print(f"⚠️ 이미지 로드 에러 {f}: {e}")
    return images, [str(f.name) for f in files]


def extract_json(text):
    """LLM 출력(Raw String)에서 JSON 객체 추출"""
    try:
        # 1. 마크다운 코드 블록 제거 (```json ... ```)
        if "```" in text:
            pattern = r"```(?:json)?\s*(\{.*?\})\s*```"
            match = re.search(pattern, text, re.DOTALL)
            if match:
                text = match.group(1)

        # 2. 앞뒤 공백 제거
        text = text.strip()

        # 3. JSON 파싱 시도
        return json.loads(text)
    except Exception:
        # 파싱 실패 시 None 반환
        return None


def generate_caption(frames, prompt):
    """캡션 생성 함수"""
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "video"},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    formatted_prompt = processor.apply_chat_template(
        conversation, add_generation_prompt=True
    )

    inputs = processor(
        text=formatted_prompt,
        videos=[frames],
        return_tensors="pt",
        padding=True,
    ).to(model.device, dtype=torch.float16)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=2000,  # JSON 출력을 위해 토큰 수 넉넉하게
            do_sample=True,
            temperature=0.5,
        )

    generated_text = processor.batch_decode(
        output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )[0]

    # 프롬프트 제외하고 응답만 추출 (후처리)
    if "ASSISTANT:" in generated_text:
        return generated_text.split("ASSISTANT:")[-1].strip()
    return generated_text.strip()

## 5. 실행 (캡셔닝 및 Google Drive 자동 저장)
- 처리되는 즉시 `.jsonl` 파일에 한 줄씩 추가(append)됩니다.
- **Fail-safe 전략**: JSON 파싱에 실패하면 `parsed_json`은 `null`로 두고, `raw_caption`에 원본 텍스트를 저장합니다. 나중에 GPT-4o-mini 등으로 후처리하는 것이 효율적입니다.

In [ ]:
# 캡션 프롬프트 설정 (JSON 출력 요구 및 먹방 특화 Enum 추가)
# PROMPT = """You are a professional Mukbang video storyboard analyst.

# The given images are frames sampled from a **"most replayed" segment** of a YouTube mukbang video.
# Treat all provided frames as a **single continuous cut**.

# Analyze **only what is visually observable** in the frames.
# Do **NOT** infer taste, internal feelings, or any unseen or implied outcomes.

# This analysis will be used for **storyboard-level visual event extraction**.

# ### Important constraints (must follow strictly)
# - Frames may be **sparsely sampled** (e.g., 1 frame per second) and may include **duplicate or near-duplicate images**.
# - Do **NOT** assume that an action is completed unless the **visual result is explicitly observable** in the frames.
# - If an action is initiated (e.g., chopsticks touching food, pressure applied) but the outcome is not visible, classify it as an **attempted or pre-action stage**, not a completed action.
# - Spoken captions or on-screen text (e.g., “탁 터뜨려서”) may reference actions that are **not visually confirmed**.
#   Treat captions as **supportive but non-authoritative evidence**.
# - Prefer uncertainty labels such as **"not_visually_verified"**, **"partial"**, or **"unknown"** over speculative completion.
# - Do **NOT** upgrade a payoff (e.g., egg yolk break, sauce flow) unless the **resulting visual change is clearly visible**.

# ### Task
# Extract **storyboard-usable visual events**, focusing on:
# - food interaction
# - visual payoff structure (setup vs. payoff)
# - replay motivation driven by anticipation or visible texture change

# Return **ONLY a single JSON object**.
# No explanations. No markdown. JSON only.

# All categorical fields **MUST strictly use one of the allowed values**.

# ### JSON schema
# {
#   "cut_type": one of ["reaction-cut", "presentation-cut", "action-cut", "transition-cut"],
#   "shot_composition": {
#     "shot_size": one of ["extreme-close-up", "close-up", "medium", "wide"],
#     "camera_angle": one of ["eye-level", "low-angle", "high-angle"],
#     "framing_focus": one of ["face", "mouth", "food", "hands", "table", "multiple"]
#   },
#   "scene_context": {
#     "location_type": one of ["home kitchen", "restaurant", "studio", "unknown"],
#     "table_setup": one of ["single-dish", "multiple-dishes", "challenge-style"],
#     "food_state_overall": one of ["intact", "partially-eaten", "mixed"]
#   },
#   "food_interaction_event": {
#     "action_type": one of ["cutting", "piercing", "breaking", "mixing", "lifting", "stretching", "biting"],
#     "tool_used": one of ["chopsticks", "fork", "spoon", "hands"],
#     "food_state_change": one of [
#       "intact",
#       "opened",
#       "broken",
#       "mixed",
#       "flowing",
#       "stringy",
#       "not_visually_verified"
#     ]
#   },
#   "visual_payoff": {
#     "setup_present": true or false,
#     "payoff_moment": one of [
#       "egg-yolk-break",
#       "first-bite",
#       "sauce-pour",
#       "cheese-pull",
#       "steam-release",
#       "none"
#     ],
#     "payoff_visibility": one of ["clear", "partial", "obstructed"]
#   },
#   "observable_emotion": one of ["neutral", "enjoyment", "surprise", "pain", "unknown"],
#   "replay_reason": {
#     "primary_trigger": one of [
#       "texture-change",
#       "precise-action",
#       "anticipation-release",
#       "sensory-synchronization"
#     ],
#     "viewer_expectation": "short factual phrase"
#   },
#   "caption_alignment": {
#     "spoken_action_match": true or false,
#     "key_phrase_candidate": "short phrase or null"
#   },
#   "scene_description_text": "concise visual description of the cut"
# }
# """
# context = "문맥: 쯔양(Tzuyu)이 을지로 대성식품 라면 햄부침집에서 다양한 음식을 즐기며, 특히 Jjapaghetti 면과 계란, 햄의 조합을 맛있게 먹는 장면입니다. 그녀는 음식이 너무 맛있어서 천천히 먹으라고 당부하며, 날씨가 좋아 야외 테이블에서 식사하고 싶다는 생각을 표현합니다. 또한, 햄이 짜지 않아 김치와 잘 어울리며, 전체적인 요리가 훌륭하다고 평가합니다.\n\n[06:22 ~ 08:25] Pop the yolk\nLooks tasty\n[1 fried egg as a whole]\nThis should go well with ham\nHam and jjapaghetti combination is incredible\nKimchi! Thank you!\nThat looks good Can't forget kimchi when eating jjapaghetti\nIt's good\n[I know it's good, but slow down!]\nThis is no joke\nToday's weather is so nice\nSo you can get a table outdoors\nI'll probably do that next time I come here\nPut it like this because the ham is not salty\nKimchi tastes good, so the dish can't go wrong\nI had to warm up everything again because the food got a little cold"


# PROMPT = f"""
# You are a professional Mukbang video storyboard analyst.

# # Input:
# - A chronological sequence of video frames from a 'most replayed' segment.
# - Weak context captions (provided below) are only guidance:
# {context}

# # Instructions:
# 1. Analyze the video as a continuous flow from beginning to end. Do NOT artificially segment into scenes.
# 2. Structure the output as a JSON object with the following keys.
# 3. Each key's value must be a fully descriptive, continuous narrative. Do not use lists or bullet points. Describe all observable motions, interactions, and visual changes over time in as much detail as possible.
# 4. Use context captions only as guidance; do not copy, paraphrase, or infer intentions, emotions, or off-frame actions.

# # JSON FORMAT (fill in actual observations):
# {{
#   "camera": "Describe the shot size, camera angle, camera height, and camera motion including static, pan, tilt, tracking, zoom in/out, and any slow or fast motion. Narrate how the camera shifts focus over time between humans, food, and environment. Include the movement path, direction, and any changes in framing or perspective as the video progresses, continuously.",
#   "humans": "Describe all people present, including full or partial visibility, facial expressions, gaze direction, hand and arm movements, posture, interactions with food or objects, and how these behaviors evolve over the segment. Include micro-movements, timing of gestures, and the continuous flow of actions as they interact with the food and surroundings.",
#   "food": "Describe all visible food items, including their placement on the table, orientation, texture, color, cooking state, layering or arrangement, and any changes over time. Narrate how humans interact with the food, how portions move, how the visual emphasis shifts, and any transformations observed during the flow.",
#   "environment": "Describe the background, props, reflections, occlusions, depth of field, table settings, visible decor, and any changes in the environment. Include interactions of humans or food with the environment and spatial relationships, describing how these evolve over time.",
#   "lighting_and_color": "Describe light sources, intensity, direction, warmth or coolness, shadows, color palette, contrast, texture impressions, and any dynamic changes during the segment. Narrate how lighting interacts with humans, food, and environment, emphasizing observable changes and visual effects.",
#   "attention_and_transitions": "Describe how camera movements, framing, composition, and visual cues direct viewer attention. Narrate where focus shifts, how transitions occur, and how the eye is guided through humans, food, and environment throughout the flow, continuously.",
#   "key_visual_elements": "Provide a single continuous text string summarizing the most visually observable elements that likely contribute to the segment's high replay value. Focus only on what is seen, not inferred intentions or off-frame context."
# }}

# # Output Rules:
# - Return only a single JSON object.
# - Ensure each value narrates the full flow from beginning to end in continuous descriptive text.
# """


# PROMPT = f"""
# You are a 30-year veteran Mukbang video director and analyst, currently studying the most replayed segments of a top-performing Mukbang YouTuber. Your goal is to understand **exactly what drives viewers to replay a segment**, focusing on observable visual elements, actions, and camera work.

# # Input:
# - A chronological sequence of video frames from a 'most replayed' segment.
# - Weak context captions (provided below) are automatically generated subtitles around the segment. Use them actively to **identify actions, food interactions, facial expressions, and timing**, but do NOT infer off-screen intentions:
# {context}

# # Instructions:
# 1. Analyze the entire segment as a single continuous flow. Do NOT split it into multiple JSON objects or time slices.
# 2. Focus on **the most engagement-driving details that likely caused viewers to replay this segment**.
# 3. For each key, describe **concrete, observable actions and visuals**:
#    - Camera: shot size, angle, height, motion (static, pan, tilt, tracking), zoom, and focus shifts between the YouTuber, food, and environment. Specify **which foods or gestures are highlighted**.
#    - Humans: YouTuber's visibility, facial expressions, gaze, hand/arm movements, posture, and interactions with food. Note **timing and sequence of key gestures**, referencing context captions.
#    - Food: visible food items, placement, arrangement, texture, color, and cooking state. Describe **how the food is manipulated and reacted to by the YouTuber**.
#    - Environment: background, props, reflections, occlusions, table setup, and spatial layout. Mention **elements that enhance focus on YouTuber and food**.
#    - Lighting_and_color: light sources, intensity, warmth/coolness, shadows, color palette, contrast, and textures. Explain **how they emphasize food and actions that likely attract replay attention**.
#    - Attention_and_transitions: camera framing, composition, motion, focus shifts, and transitions. Highlight **moments where the viewer's gaze is guided to food or gestures**.
#    - Key_visual_elements: summarize **the single most compelling visual elements and moments** in the segment, including food interactions, hand movements, facial expressions, and camera work.
#    - Why_replayed: provide **concrete reasons why viewers likely replayed this segment**, linking them to specific foods, gestures, expressions, or camera emphasis.

# # Output format:
# Return a single JSON object like this:

# {{
#   "camera": "...",
#   "humans": "...",
#   "food": "...",
#   "environment": "...",
#   "lighting_and_color": "...",
#   "attention_and_transitions": "...",
#   "key_visual_elements": "...",
#   "why_replayed": "..."
# }}

# # Additional rules:
# - Avoid abstract adjectives like 'warm', 'pleasant' without linking them to **observable actions or visuals**.
# - Include **food names, gestures, and specific expressions** whenever they appear in the context captions.
# - Describe sequences in **time order**, but focus only on **the most engagement-driving moments**.
# - Always refer to the YouTuber as 'the YouTuber', never 'chef'.
# """

PROMPT = f"""
You are a 30-year veteran Mukbang video director and analyst, currently studying the most replayed segments of a top-performing Mukbang YouTuber. Your task is to analyze the segment **strictly based on the provided video frames**. Context captions (automatic subtitles) are provided only as timing references for actions or speech, and should **not** be used to infer off-screen intentions or emotions. 

# Input:
- A chronological sequence of video frames from a 'most replayed' segment.
- Context captions for timing guidance: {context}

# Instructions:
1. Analyze the entire segment as a **continuous narrative**. Do NOT generate separate objects per frame or time slice. 
2. Focus on **what is visually observable in each frame**: camera shot, angle, motion, zoom, lighting, background, food position, hand movements, facial expressions, gaze, posture, and interactions with food or props.
3. Only describe what you can **directly see in the frames**. Do **not** assume feelings, intentions, or actions not visible.
4. Emphasize **elements that likely make viewers replay the segment**, including specific camera framing, food presentation, or visually satisfying motions.
5. Integrate the context captions **only to reference when specific actions or speech occur**, not to invent content.

# Output Format:
{{
  "narrative": "Provide a long, continuous, detailed description of the segment that integrates all key observable elements: camera work, human actions, food, environment, lighting, and attention cues. Explain why certain visual elements might drive replay, referencing context captions only for timing. Focus on concrete visual details (e.g., camera is a tight overhead shot, YouTuber lifts kimchi with chopsticks, ham slice glistens under warm light, hand hovers above plate before picking up food, etc.) and avoid abstract/generalized descriptions."
}}
"""

# 대상 폴더 스캔
frames_path = Path(FRAMES_DIR)
if not frames_path.exists():
    print(f"❌ 오류: 데이터 폴더가 없습니다 -> {frames_path}")
else:
    video_folders = sorted([d for d in frames_path.iterdir() if d.is_dir()])[:5]
    print(f"총 {len(video_folders)}개의 비디오 폴더를 찾았습니다.")

    for video_folder in tqdm(video_folders, desc="Processing Videos"):
        video_id = video_folder.name
        output_file = Path(OUTPUT_DIR) / f"{video_id}.jsonl"

        # 이미 처리된 ID 확인 (Resume 기능)
        processed_ids = set()
        if output_file.exists():
            with open(output_file, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        try:
                            data = json.loads(line)
                            processed_ids.add((data["recollect_id"], data["rank"]))
                        except:
                            pass

        # Recollect ID 순회
        for recollect_folder in sorted(video_folder.iterdir()):
            if not recollect_folder.is_dir():
                continue

            try:
                recollect_id = int(recollect_folder.name)
            except:
                continue

            # 세그먼트 순회
            segments = sorted(recollect_folder.iterdir())
            for segment_folder in segments:
                if not segment_folder.is_dir():
                    continue

                meta = parse_segment_folder(segment_folder.name)
                if not meta:
                    continue

                # 이미 처리했으면 스킵
                if (recollect_id, meta["rank"]) in processed_ids:
                    continue

                # 이미지 로드
                frames, filenames = load_frames(segment_folder)
                if not frames:
                    continue

                print(
                    f"📸 처리 중: {video_id} / {recollect_id} / {meta['rank']} ({len(frames)}장)"
                )

                try:
                    # 1. 모델 추론
                    raw_output = generate_caption(frames, PROMPT)

                    # 2. JSON 파싱 시도 (markdown code block 지원)
                    parsed_json = extract_json(raw_output)

                    if parsed_json:
                        print(f"   ✅ JSON 파싱 성공")
                    else:
                        # 파싱 실패해도 진행 (Raw Text 저장)
                        print(f"   ⚠️ JSON 파싱 실패 -> Raw Text 저장")

                    # 3. 결과 구성
                    result = {
                        "video_id": video_id,
                        "recollect_id": recollect_id,
                        "rank": meta["rank"],
                        "start_sec": meta["start_sec"],
                        "end_sec": meta["end_sec"],
                        "file_names": filenames,
                        "parsed_json": parsed_json,  # 성공 시 dict, 실패 시 null
                        "raw_caption": raw_output,  # 원본 텍스트 (Backup)
                    }

                    # 4. 즉시 저장 (append mode)
                    with open(output_file, "a", encoding="utf-8") as f:
                        f.write(json.dumps(result, ensure_ascii=False) + "\n")

                except Exception as e:
                    print(f"❌ 에러 발생: {e}")